# EDA 06: Enterprise Data Quality, Missingness & Outlier Audit

This notebook performs a comprehensive data quality audit across all 8 processed tables, evaluating completeness, primary key uniqueness, data type integrity, and extreme value distributions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

tables = ['sales_fact', 'product_dim', 'warehouse_dim', 'supplier_dim', 'promotion_dim', 'calendar_dim', 'weather_dim', 'event_dim']
table_dfs = {t: pd.read_parquet(f'data/processed/{t}.parquet') for t in tables}

print("Loaded all 8 processed tables successfully.")


Loaded all 8 processed tables successfully.


In [2]:
# Table Completeness Audit (Null Percentage per Column)
null_audit = []
for name, df in table_dfs.items():
    total_rows = len(df)
    null_cols = df.isnull().sum()
    for col, null_cnt in null_cols.items():
        null_pct = (null_cnt / total_rows) * 100
        null_audit.append({
            'table_name': name,
            'column_name': col,
            'dtype': str(df[col].dtype),
            'total_rows': total_rows,
            'null_count': null_cnt,
            'null_pct': round(null_pct, 4)
        })

df_null_audit = pd.DataFrame(null_audit)
print("Columns with any missing values:")
print(df_null_audit[df_null_audit['null_count'] > 0])
if (df_null_audit['null_count'] == 0).all():
    print("ALL 8 TABLES HAVE 0% NULL VALUES ACROSS ALL COLUMNS!")


Columns with any missing values:
Empty DataFrame
Columns: [table_name, column_name, dtype, total_rows, null_count, null_pct]
Index: []
ALL 8 TABLES HAVE 0% NULL VALUES ACROSS ALL COLUMNS!


In [3]:
# Primary Key Uniqueness Audit
pk_map = {
    'sales_fact': 'sales_id',
    'product_dim': 'product_id',
    'warehouse_dim': 'warehouse_id',
    'supplier_dim': 'supplier_id',
    'promotion_dim': 'promotion_id',
    'calendar_dim': 'date',
    'weather_dim': 'weather_id',
    'event_dim': 'event_id'
}

pk_audit = []
for name, df in table_dfs.items():
    pk_col = pk_map[name]
    total_count = len(df)
    unique_count = df[pk_col].nunique()
    dupe_count = total_count - unique_count
    pk_audit.append({
        'table_name': name,
        'primary_key': pk_col,
        'total_rows': total_count,
        'unique_keys': unique_count,
        'duplicate_keys': dupe_count,
        'is_unique': dupe_count == 0
    })

df_pk_audit = pd.DataFrame(pk_audit)
print("Primary Key Audit Summary:")
print(df_pk_audit)


Primary Key Audit Summary:
      table_name   primary_key  ...  duplicate_keys  is_unique
0     sales_fact      sales_id  ...               0       True
1    product_dim    product_id  ...               0       True
2  warehouse_dim  warehouse_id  ...               0       True
3   supplier_dim   supplier_id  ...               0       True
4  promotion_dim  promotion_id  ...               0       True
5   calendar_dim          date  ...               0       True
6    weather_dim    weather_id  ...               0       True
7      event_dim      event_id  ...               0       True

[8 rows x 6 columns]


In [4]:
# Outlier & Skewness Audit for Sales Fact Metrics
df_sales = table_dfs['sales_fact']
numeric_cols = ['total_sales', 'quantity', 'unit_price', 'shipping_cost', 'profit']

num_audit = []
for col in numeric_cols:
    s = df_sales[col]
    num_audit.append({
        'metric': col,
        'min': s.min(),
        'p01': s.quantile(0.01),
        'p50 (median)': s.median(),
        'mean': s.mean(),
        'p99': s.quantile(0.99),
        'max': s.max(),
        'std': s.std(),
        'skewness': s.skew()
    })

df_num_audit = pd.DataFrame(num_audit)
print("Numerical Distribution & Winsorization Audit:")
print(df_num_audit)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_pk_audit, x='table_name', y='total_rows', palette='crest')
plt.title('Total Record Count per Unified Table')
plt.ylabel('Row Count (Log Scale)')
plt.yscale('log')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


Numerical Distribution & Winsorization Audit:
          metric         min         p01  ...           max           std   skewness
0    total_sales     9.99000   23.900000  ...  2.404035e+06  88008.296653  17.509434
1       quantity     1.00000    1.000000  ...  7.388000e+03    480.396624   1.473777
2     unit_price     0.85000    5.458742  ...  3.818686e+06  89007.579886  17.892306
3  shipping_cost     0.00000    0.000000  ...  4.096800e+02      7.751124   8.789015
4         profit -4274.97998 -101.250000  ...  9.118000e+02     42.252118 -10.019216

[5 rows x 9 columns]
